In [10]:
import numpy as np

from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy

In [11]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 50,
    "batch_size": 64,
    "loss": CategoricalCrossentropy(),
    "optimizer": SGD(learning_rate=0.001, momentum=0.9)
}

In [12]:
from keras.datasets import cifar10
from keras.utils import to_categorical
# load dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
# one hot encode target values
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
# normalize to range 0-1
X_train = X_train / 255.0
X_test = X_test / 255.0


np.save('train_data.npy', X_train)
np.save('train_labels.npy', y_train)
np.save('test_data.npy', X_test)
np.save('test_labels.npy', y_test)

In [13]:
def get_train_data():
    return np.load('train_data.npy'), np.load('train_labels.npy')

def get_test_data():
    return np.load('test_data.npy'), np.load('test_labels.npy')

In [14]:
def create_model():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu', kernel_initializer='he_uniform'))
    model.add(Dropout(0.2))
    model.add(Dense(10, activation='softmax'))
    return model

def train_model(model, X_train, y_train, loss, optimizer, epochs, batch_size):
    model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size)
    return model

In [15]:
import tensorflow as tf

# Check for GPU availability
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Print the device name
print("Device Name: ", tf.test.gpu_device_name())


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("Using GPU: ", tf.test.gpu_device_name())
    except RuntimeError as e:
        print(e)

Num GPUs Available:  1
Device Name:  /device:GPU:0
Using GPU:  /device:GPU:0


In [16]:
X_train, y_train = get_train_data()

In [17]:
model = create_model()
model = train_model(model, X_train, y_train, config["loss"], config["optimizer"], config["epochs"], config["batch_size"])

Epoch 1/50
782/782 [==============================] - 8s 9ms/step - loss: 2.0181 - accuracy: 0.2468
Epoch 2/50
782/782 [==============================] - 7s 9ms/step - loss: 1.6559 - accuracy: 0.3880
Epoch 3/50
782/782 [==============================] - 7s 9ms/step - loss: 1.5065 - accuracy: 0.4450
Epoch 4/50
782/782 [==============================] - 7s 9ms/step - loss: 1.4169 - accuracy: 0.4831
Epoch 5/50
782/782 [==============================] - 7s 9ms/step - loss: 1.3386 - accuracy: 0.5147
Epoch 6/50
782/782 [==============================] - 7s 9ms/step - loss: 1.2738 - accuracy: 0.5399
Epoch 7/50
782/782 [==============================] - 7s 9ms/step - loss: 1.2086 - accuracy: 0.5658
Epoch 8/50
782/782 [==============================] - 7s 9ms/step - loss: 1.1573 - accuracy: 0.5850
Epoch 9/50
782/782 [==============================] - 7s 9ms/step - loss: 1.1072 - accuracy: 0.6057
Epoch 10/50
782/782 [==============================] - 7s 9ms/step - loss: 1.0615 - accuracy: 0.6201

In [18]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

157/157 [==============================] - 1s 4ms/step - loss: 0.6002 - accuracy: 0.7995

Test accuracy: 79.9%
